# Cleaning downloaded Data from GISAID

Author: Alexander Maksiaev

Purpose: Download and clean GISAID data, after de-duplicating from Andersen/NCBI Virus data

Notes: 
* The "downloads" folder MUST be your computer's downloads folder, or wherever your browser automatically downloads files. This folder must also be cleaned in between each run of this code.
* The returned files from this code will be stored in a separate folder after running -- no other action is needed, aside from cleaning the original downloads folder after this code runs.  
* This file MUST be in the same folder as "utils.py"

## Housekeeping ##

In [1]:
import os
import shutil
import pandas as pd
import numpy as np
import dateutil
import openpyxl
from itertools import islice
import importlib
import utils  
importlib.reload(utils)
from utils import * 

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [102]:
# Paths

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
# downloads = "C:/Users/maksi/Downloads/"
# references = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/"
references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references"
downloads = "C:/Users/maksiaevai.NCBI_NT/Downloads/" # MUST be downloads folder. Clean out downloads folder after each use. 
andersen_ncbi_virus_gisaid = home + "Combinations/NCBI_Virus_Andersen_GISAID/" 

# Collect user input

# locations = input("Locations (separate with commas and no spaces): ")
# start_date = input("Start date (format: YYYY-MM-DD): ")
# end_date = input("End date (format: YYYY-MM-DD): ")

locations = "Antarctica,North America,South America"
start_date = "2021-11-01"
end_date = "2026-02-20"
date_range = dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y")

os.chdir(downloads)

# Create directories if needed
downloads_saved = home + "GISAID/downloads/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
andersen_ncbi_virus = home + "Combinations/NCBI_Virus_Andersen/" + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
gisaid_files = home + "GISAID/complete/" + start_date + "--" + end_date + "_" + locations.replace(",", "_").replace(" ", "_") + "/"
complete_files = andersen_ncbi_virus_gisaid + dateutil.parser.parse(start_date).strftime("%m-%d-%Y") + "--" + dateutil.parser.parse(end_date).strftime("%m-%d-%Y") + "_" + locations.replace(",", "_").replace(" ", "_") + "/"

if not os.path.exists(gisaid_files): # checking if the directory exists or not
    os.makedirs(gisaid_files) # if the directory is not present then create it

if not os.path.exists(complete_files): # checking if the directory exists or not
    os.makedirs(complete_files) # if the directory is not present then create it

# Get list of genotypes and states

# os.chdir("C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu/references/")
os.chdir(references)

states = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")

# genotypes = list(genotypes_df["Genotype"])
# genotypes.append("Unassigned")

# print(genotypes)

genotypes = ["B3.13", "D1.1", "D1.3", "Not"] 

## Download all files, convert fasta files to dataframes

In [3]:
all_metadata_files = []
all_fasta_files = []

if not os.path.exists(downloads_saved): # checking if the directory exists or not
    os.makedirs(downloads_saved) # if the directory is not present then create it

# Move downloaded files to saved downloads
for dirpath, dirs, files in os.walk(downloads):
    if len(files) > 0: # If we have any files that need to be moved
        for file in files:
            file_name = os.path.join(dirpath, file)
            destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
            try:
                shutil.move(file_name, destination_path)
            except:
                print("Error moving file", file_name)
                continue 
    elif len(files) == 0 and len(os.listdir(downloads_saved)) == 0: # If we don't have any downloaded files
        # Have user type in username and password
        username = input("Username: ")
        password = input("Password: ")
        browser = input("Browser: ")
        sleep_time = input("Seconds to sleep in between clicks (recommended 5): ")

        open_gisaid(username, password, browser, sleep_time, locations, start_date, end_date) # Download files -- NOT WORKING RIGHT NOW
        # First batch: 2021-11-01 -- 2024-12-31
        # Second batch: 2025-01-01 -- present

        # Re-try 
        for dirpath, dirs, files in os.walk(downloads):
            if len(files) > 0: # If we have any files that need to be moved
                for file in files:
                    file_name = os.path.join(dirpath, file)
                    destination_path = os.path.join(downloads_saved, os.path.basename(file_name))
                    try:
                        shutil.move(file_name, destination_path)
                    except:
                        print("Error moving file", file_name)
                        continue 
            break 
    else: # If we have downloaded files saved already
        continue
    break 

# If we have multiple files, name them nicely
for dirpath, dirs, files in os.walk(downloads_saved):
    for file in files:
        file_name = os.path.join(dirpath, file)
        # file_name = "_".join(file_name.split(" "))
        
        
        # os.rename(file_name, "_".join(file_name.split(" ")).replace("(", "").replace(")", ""))
        # file_name = "_".join(file_name.split(" ")).replace("(", "").replace(")", "")

        print(file_name)

        # Now go through files and get contents
        if ".xls" in file_name:
            metadata = pd.read_excel(file_name)
            all_metadata_files.append(metadata)
        if ".fasta" in file_name:
            fasta_file = fasta_df(file_name, states) # Convert fasta file to dataframe
            all_fasta_files.append(fasta_file)
    break 

C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-02-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates.xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-02-20_Antarctica_North_America_South_America/gisaid_epiflu_isolates_1.xls
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-02-20_Antarctica_North_America_South_America/gisaid_epiflu_sequence.fasta
C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/GISAID/downloads/2021-11-01--2026-02-20_Antarctica_North_America_South_America/gisaid_epiflu_sequence_1.fasta


In [4]:
# Concatenate metadata

metadata_concat = pd.DataFrame()
for metadata_file in all_metadata_files:
    metadata_concat = pd.concat([metadata_concat, metadata_file])

In [ ]:
# Function to get metadata
def separate_fasta_by_segments(metadata, fasta, animals_df, genotypes): #, b313_fasta, d11_fasta):

    fasta = fix_animals(fasta, animals_df) # Fix animals first

    unique_segments = list(set(fasta["Segment"])) # Get list of segments

    # “>EPI_ID|Isolate_name|subtype|collection_date|host_type|genotype”

    segment_fastas = [] # Get a list of fastas, separated by segment
    for genotype in genotypes: # .keys(): # For each genotype
        for seg in unique_segments: # For each segment
            print(list(set(metadata["Genotype"])))
            xls = metadata[metadata["Genotype"].str.contains(genotype)] # [metadata["Genotype"].apply(lambda x: x.split(" ")[0]) == genotype] # Get only the metadata corresponding to that genotype
            xls = xls.rename(columns={"Isolate_Id":"Identifier"})
            # print(xls)
            # print("XLS: ", metadata["Genotype"])
            # print(xls["Clade"])

            # fasta_seg_pre = fasta[fasta["Identifier"].isin(xls['Isolate_Id'])] # Get only the identifiers (Isolate_Id) that are left after metadata is filtered for genotype
            fasta_seg_pre = fasta.merge(xls, how="right", on="Identifier")
            # print(fasta_seg_pre.columns)
            # break 

            # fasta_seg = genotype_fastas[fasta_gen][genotype_fastas[fasta_gen]["Segment"] == seg]
            fasta_seg = fasta_seg_pre[fasta_seg_pre["Segment"] == seg]

            fasta_seg["Genotype"] = genotype

            # Rename sequences 
            new_name = ">" + fasta_seg["Identifier"] + "|" + fasta_seg["Isolate_Name_x"] + "|" + fasta_seg["Subtype_x"] + "|" + fasta_seg["Geo_Location"] + "|" + fasta_seg["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day else x) + "|" + fasta_seg["Host_Type"] + "|" + fasta_seg["Genotype"] #.apply(lambda x: "" if x != "human" else "|human")
            fasta_seg["full_header"] = new_name
            # print(fasta_seg["New_Name"])

            segment_fastas.append(fasta_seg)
            print(fasta_seg[["full_header", "Header", "Isolate_Id", "Isolate_Name_x", "Subtype_x", "Segment", "Geo_Location", "Date Collected", "Identifier", "Host_Type", "Isolate_Name_y", "Subtype_y", "Genotype", "Location", "Collection_Date"]])

            # segment_fastas.append(fasta_seg)

    return segment_fastas, unique_segments

In [103]:
# Separate fastas by segment -- results in number of downloaded fastas * number of genotypes * 8 segments
segment_fastas = []
unique_animals_all = []
for i, fasta in enumerate(all_fasta_files):

    metadata = metadata_concat

    unique_animals = sort_animals(fasta) # Find unique animals
    # print("Animals: ", unique_animals)
    unique_animals_all.append(unique_animals)

    os.chdir(references)
    animals_ref = pd.read_csv("animals_ref.csv")

    fastas, unique_segments = separate_fasta_by_segments(metadata, fasta, animals_ref, genotypes) # Separate the fasta dataframes into 8 different files based on segment

    for fasta in fastas:
        segment_fastas.append(fasta)

# print(segment_fastas) # [0][0][segment_fastas[0][0]["Genotype"] == "D1.1"])

["Minor70 (<i style='font-size:11px'>GenoFLU</i>)", "B3.2 (<i style='font-size:11px'>GenoFLU</i>)", "D1.1 (<i style='font-size:11px'>GenoFLU</i>)", "Minor89 (<i style='font-size:11px'>GenoFLU</i>)", "A2 (<i style='font-size:11px'>GenoFLU</i>)", "Minor51 (<i style='font-size:11px'>GenoFLU</i>)", "Minor81 (<i style='font-size:11px'>GenoFLU</i>)", "A6 (<i style='font-size:11px'>GenoFLU</i>)", "Minor33 (<i style='font-size:11px'>GenoFLU</i>)", "A4 (<i style='font-size:11px'>GenoFLU</i>)", "C3.1 (<i style='font-size:11px'>GenoFLU</i>)", "Notassigned (<i style='font-size:11px'>GenoFLU</i>)", "Minor61 (<i style='font-size:11px'>GenoFLU</i>)", "Minor76 (<i style='font-size:11px'>GenoFLU</i>)", "A5 (<i style='font-size:11px'>GenoFLU</i>)", "Minor35 (<i style='font-size:11px'>GenoFLU</i>)", "Minor73 (<i style='font-size:11px'>GenoFLU</i>)", "Minor99 (<i style='font-size:11px'>GenoFLU</i>)", "Minor65 (<i style='font-size:11px'>GenoFLU</i>)", "B3.9 (<i style='font-size:11px'>GenoFLU</i>)", "B1.2 (

In [104]:
print(len(segment_fastas))

64


## De-Duplication

In [105]:
# Get files from Andersen and NCBI Virus

# Grab files
andersen_ncbi = {} # Results in number of genotypes * 8 segments
for dirpath, dirs, files in os.walk(andersen_ncbi_virus):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
        # print(file_name)
            segment_genotype = "_".join(file_name.split("/")[-1].split("_")[0:2])
            fasta_file = fasta_df_complete(file_name, states) # Convert fasta file to dataframe
            andersen_ncbi[segment_genotype] = fasta_file

________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly______________________________________GenBank_Title__\
________Accession_Assembly__

In [106]:
print(andersen_ncbi)

{'B3.13_HA':                                                  Header     Isolate_Id  \
0     SRR28752446|A/blackbird/Texas/24-008354-001/20...  24-008354-001   
1     SRR28752447|A/cattle/Texas/24-009108-005/2024|...  24-009108-005   
2     SRR28752448|A/cattle/Texas/24-009108-004/2024|...  24-009108-004   
3     SRR28752449|A/cattle/Texas/24-009108-003/2024|...  24-009108-003   
4     SRR28752450|A/cattle/Texas/24-009108-002/2024|...  24-009108-002   
...                                                 ...            ...   
5130  GCA_054872375|A/Bovine/California/B2400548/202...       B2400548   
5131  GCA_054872415|A/Bovine/California/B2401408/202...       B2401408   
5132  GCA_054872425|A/Bovine/California/B2401620/202...       B2401620   
5133  GCA_054872495|A/Bovine/California/B2401751/202...       B2401751   
5134  GCA_054872785|A/Turkey/California/B2402661/202...       B2402661   

                              Isolate_Name Subtype    Partials    Location  \
0     A/blackbird/Te

### Collect partial isolates from all parties

In [181]:
# Do all segments, not just HA 

# Check isolate IDs to see if they already exist in Andersen/NCBI
# Match based on year AND partial isolate ID, as some partials may be identical between years

# GISAID partial isolates
gisaid_list = {}
for gisaid_fasta in segment_fastas:
    # for genotype_gisaid_fasta in gisaid_fasta:
    # print(genotype_gisaid_fasta)
    gisaid_fasta["Partials"] = gisaid_fasta["Isolate_Id"].apply(partial_isolate)
    gisaid_fasta["Year"] = gisaid_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x).year))
    genotype_gisaid_fasta = gisaid_fasta.drop_duplicates(subset=["Partials", "Year", "Segment"], keep="first")
    if genotype_gisaid_fasta["Genotype"].values[0] not in gisaid_list.keys():
        gisaid_list[genotype_gisaid_fasta["Genotype"].values[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]] = genotype_gisaid_fasta
    else:
        gisaid_list[genotype_gisaid_fasta["Genotype"].values[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]] = pd.concat([gisaid_list[genotype_gisaid_fasta["Genotype"].values[0] + "_" + genotype_gisaid_fasta["Segment"].values[0]], genotype_gisaid_fasta]).drop_duplicates(subset=["Partials", "Year", "Segment"], keep="last")
    
    # print(genotype_gisaid_fasta)
        
# NCBI_Virus/Andersen partial isolates    
andersen_ncbi_genotypes = {} # Results in # of genotypes
for key in andersen_ncbi:
    print(key)
    andersen_ncbi_fasta = andersen_ncbi[key] 

    # Find partial Isolate IDs -- humans and non-humans have different locations for isolates
    andersen_ncbi_fasta_nonhuman = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] != "human"]
    andersen_ncbi_fasta_nonhuman["Partials"] = andersen_ncbi_fasta_nonhuman["Isolate_Id"].apply(partial_isolate) # For non-humans, apply partial function to isolate ids

    andersen_ncbi_fasta_human = andersen_ncbi_fasta[andersen_ncbi_fasta["Host_Type"] == "human"]
    # andersen_ncbi_fasta_human["Partials_Prep"] = andersen_ncbi_fasta_human["Isolate_Name"].apply(lambda x: x.split("/")[-2]) # For humans, apply partial function to location, because isolate comes earlier
    andersen_ncbi_fasta_human["Partials"] = andersen_ncbi_fasta_human["Isolate_Id"].apply(partial_isolate)
    # andersen_ncbi_fasta["Partials"] = andersen_ncbi_fasta["Isolate_Id"].apply(partial_isolate)
    # print("Andersen:", andersen_ncbi_fasta["Partials"])

    # Concatenate humans and non-humans
    andersen_ncbi_fasta = pd.concat([andersen_ncbi_fasta_human, andersen_ncbi_fasta_nonhuman])
    # print(andersen_ncbi_fasta)

    # Get year and segment
    andersen_ncbi_fasta["Year"] = andersen_ncbi_fasta["Date Collected"].apply(lambda x: str(dateutil.parser.parse(x, default=dateutil.parser.parse("2000-01-01")).year) if len(x) > 0 else x)
    andersen_ncbi_fasta["Segment"] = key.split("_")[-1]
    # andersen_ncbi_fasta = andersen_ncbi_fasta.drop_duplicates(subset=["Partials", "Year"], keep="first")
    # print(andersen_ncbi_fasta)

    if andersen_ncbi_fasta["Genotype"].values[0] not in andersen_ncbi_genotypes.keys(): # If we haven't already seen this genotype
        andersen_ncbi_genotypes[andersen_ncbi_fasta["Genotype"].values[0].split(" ")[0] + "_" + andersen_ncbi_fasta["Segment"].values[0]] = andersen_ncbi_fasta # Add fasta to genotype (To include "Not" assigned) dictionary
        # print(andersen_ncbi_fasta)
    # break 


B3.13_HA
B3.13_MP
B3.13_NA
B3.13_NP
B3.13_NS
B3.13_PA
B3.13_PB1
B3.13_PB2
D1.1_HA
D1.1_MP
D1.1_NA
D1.1_NP
D1.1_NS
D1.1_PA
D1.1_PB1
D1.1_PB2
D1.3_HA
D1.3_MP
D1.3_NA
D1.3_NP
D1.3_NS
D1.3_PA
D1.3_PB1
D1.3_PB2


In [182]:
print(andersen_ncbi_genotypes["B3.13_PB2"]) #["Segment"].values)

                                                 Header     Isolate_Id  \
3981  SRR31840198|A/Washington/OR/24-037325-011/2024...  24-037325-011   
0     SRR28752446|A/blackbird/Texas/24-008354-001/20...  24-008354-001   
1     SRR28752447|A/cattle/Texas/24-009108-005/2024|...  24-009108-005   
2     SRR28752448|A/cattle/Texas/24-009108-004/2024|...  24-009108-004   
3     SRR28752449|A/cattle/Texas/24-009108-003/2024|...  24-009108-003   
...                                                 ...            ...   
5130  GCA_054872375|A/Bovine/California/B2400548/202...       B2400548   
5131  GCA_054872415|A/Bovine/California/B2401408/202...       B2401408   
5132  GCA_054872425|A/Bovine/California/B2401620/202...       B2401620   
5133  GCA_054872495|A/Bovine/California/B2401751/202...       B2401751   
5134  GCA_054872785|A/Turkey/California/B2402661/202...       B2402661   

                              Isolate_Name Subtype    Partials    Location  \
3981    A/Washington/OR/24-037325

In [183]:
for gisaid_df in gisaid_list:
    print(gisaid_df)
    print(gisaid_list[gisaid_df])
    print(len(gisaid_df))

print(len(gisaid_list))

B3.13_PB2
                                                  Header  Isolate_Id  \
2380   EPI_ISL_19792258|A/turkey/USA/007640-001/2025|...  007640-001   
2388   EPI_ISL_19743023|A/turkey/California/003574-00...  003574-001   
2396   EPI_ISL_19743022|A/turkey/California/003574-00...  003574-002   
2404   EPI_ISL_19743021|A/turkey/California/003574-00...  003574-003   
2412   EPI_ISL_19743020|A/turkey/California/003903-00...  003903-001   
...                                                  ...         ...   
18876  EPI_ISL_20266001|A/dairy_cow/California/026426...  026426-010   
18884  EPI_ISL_20265994|A/dairy_cow/California/02121-...   02121-002   
18892  EPI_ISL_20265992|A/dairy_cow/California/02121-...   02121-007   
18908  EPI_ISL_20265984|A/dairy_cow/California/02652-...   02652-001   
18916  EPI_ISL_20265578|A/dairy_cow/USA/024867-001/20...  024867-001   

                               Isolate_Name_x Subtype_x Segment  \
2380             A/turkey/USA/007640-001/2025      H5N1   

### Throw away duplicate isolates from GISAID

In [ ]:
gisaid_dict = {}

for gisaid_genotype in gisaid_list: 
    gisaid_genotype_df = gisaid_list[gisaid_genotype]
    print("original:", len(gisaid_genotype_df))
    print(gisaid_genotype)
    if gisaid_genotype in andersen_ncbi_genotypes.keys(): # If they share the genotype
        # Left inner on gisaid, so we can get all duplicates and ignore unique Andersen entries
        gisaid_duplicates = pd.merge(gisaid_genotype_df, andersen_ncbi_genotypes[gisaid_genotype], how="inner") #, indicator=True) #, join="left") #, on=shared_columns)
        # print(gisaid_duplicates)
        gisaid_deduplicated = pd.concat([gisaid_genotype_df, gisaid_duplicates]).drop_duplicates(subset=["Partials"], keep=False) # Use Andersen/NCBI_Virus duplicates instead of GISAID
        gisaid_dict[gisaid_genotype] = gisaid_deduplicated
        print("new: both", len(gisaid_deduplicated))
    else: # If this is a GISAID-only genotype
        gisaid_deduplicated = gisaid_genotype_df.drop_duplicates(subset=["Partials"], keep="last")
        gisaid_dict[gisaid_genotype] = gisaid_deduplicated
        print("new:", len(gisaid_deduplicated))
            

# throw_away = {} # Dictionary of isolates to throw

# counter = 0
# for gisaid_genotype in gisaid_list: # Each dataframe is unique in genotype
#     # print(gisaid_genotype)
#     if len(gisaid_genotype["Genotype"]) > 0:
#         genotype = gisaid_genotype["Genotype"].values[0]
#         andersen_ncbi_fasta = pd.DataFrame()
#         if genotype in andersen_ncbi_genotypes.keys(): # If it's in Andersen/NCBI_Virus and we haven't seen it before here
#             andersen_ncbi_fasta = andersen_ncbi_genotypes[genotype] # Get the dataframe with the same genotype
#             gisaid_duplicates = gisaid_genotype[gisaid_genotype.duplicated(["Partials", "Year"], keep=False)]
#             # print(gisaid_duplicates)
#             andersen_ncbi_fasta_duplicates = andersen_ncbi_fasta[andersen_ncbi_fasta.duplicated(["Partials", "Year"], keep=False)]
#             to_throw = pd.concat([gisaid_genotype, andersen_ncbi_fasta])[pd.concat([gisaid_genotype, andersen_ncbi_fasta]).duplicated(["Partials", "Year"], keep=False)]
            
#             between_duplicates = []
#             for t in to_throw["Identifier"].values:
#                 if t not in gisaid_duplicates and t not in andersen_ncbi_fasta_duplicates:
#                 # if t in gisaid_duplicates or t in andersen_ncbi_fasta_duplicates:
#                     # print(t)
#                     between_duplicates.append(t)
#             if genotype in throw_away.keys(): # If there's already a genotype
#                 throw_away[genotype] += between_duplicates
#             else:
#                 throw_away[genotype] = between_duplicates
#         else: # If it's a genotype not seen in Andersen/NCBI_Virus, don't throw it
#             print("GISAID genotype:", gisaid_genotype["Genotype"].values[0])
#             throw_away[genotype] = []
#     counter += 1

# for key in throw_away:
#     print(key)
#     print(len(throw_away[key]))

# kept_seqs = []
# print(len(segment_fastas))
# for gisaid_df in segment_fastas:
#     # print(len(genotype_group))
#     # for gisaid_df in genotype_group:
#         # print(gisaid_df)
#     if len(gisaid_df["Genotype"].dropna()) > 0: # If there are sequences
#         genotype = gisaid_df["Genotype"].values[0]
#         # print(len(genotype_seq_keep[genotype]))
#         # gisaid_df_new = gisaid_df[gisaid_df['Identifier'].isin(genotype_seq_keep[genotype])]

#         # Check how much was thrown away
#         if genotype in throw_away:
#             to_throw = throw_away[genotype]
#             print("Original length:", genotype, len(gisaid_df))
#             gisaid_df_new = gisaid_df[~gisaid_df['Identifier'].isin(to_throw)]
#             print("New length:", len(gisaid_df_new))
#             kept_seqs.append(gisaid_df_new)
#         # else:


# # print(len(kept_seqs))

original: 2001
B3.13_PB2
new: both 1767
original: 2001
B3.13_MP
new: both 1767
original: 2001
B3.13_NA
new: both 1767
original: 2001
B3.13_PA
new: both 1767
original: 2001
B3.13_HA
new: both 1767
original: 2001
B3.13_NS
new: both 1767
original: 2001
B3.13_PB1
new: both 1767
original: 2001
B3.13_NP
new: both 1767
original: 4326
D1.1_PB2
new: both 4256
original: 4326
D1.1_MP
new: both 4256
original: 4326
D1.1_NA
new: both 4256
original: 4326
D1.1_PA
new: both 4256
original: 4326
D1.1_HA
new: both 4256
original: 4326
D1.1_NS
new: both 4256
original: 4326
D1.1_PB1
new: both 4256
original: 4326
D1.1_NP
new: both 4256
original: 298
D1.3_PB2
new: both 290
original: 298
D1.3_MP
new: both 290
original: 298
D1.3_NA
new: both 290
original: 298
D1.3_PA
new: both 290
original: 298
D1.3_HA
new: both 290
original: 298
D1.3_NS
new: both 290
original: 298
D1.3_PB1
new: both 290
original: 298
D1.3_NP
new: both 290
original: 546
Not_PB2
new: 545
original: 546
Not_MP
new: 545
original: 546
Not_NA
new: 545

### Create animal reference if needed 

In [197]:
# Find animals to sort, if needed

os.chdir(references)

# Flatten unique_animals
every_unique_animal = []
for l in unique_animals_all:
    for animal in l:
        every_unique_animal.append(animal)

# Rename host type

unique_animals_set = list(set(every_unique_animal))
animals_df = pd.DataFrame(columns=["wild_avian", "domestic_avian", "cattle", "feline", "other_mammal", "human", "pet_food", "other"])
animals_df["other"] = unique_animals_set # to sort

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

print(common_animals)
print(len(common_animals))

different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

print(animals_ref)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer
else:
    number_of_times_to_add_nan = len(different_animals) - len(animals_df)
    nan_row = pd.DataFrame([[np.nan] * len(animals_df.columns)], columns=animals_df.columns)
    for i in range(number_of_times_to_add_nan):
        animals_df = pd.concat([animals_df, nan_row], ignore_index=True)

animals_df["new"] = (different_animals)

print(animals_df)

os.chdir(references)
animals_df.to_csv("animals_ref_to_sort.csv")

['white-winged_scoter', 'red_necked_grebe', 'american_widgeon', 'antarctic_tern', 'rock_dove', 'blue_winged_teal', 'scoter', 'surf_scoter', 'double-crested_co', 'tern', 'crow', 'south_american_fur_seal', 'brown_booby', 'black_crowned_night-heron', 'rock_goose', 'avian', 'western_screech_owl', 'crane', 'double-crested_cormorant', 'silkie_chicken', 'red_tailed_hawk', 'ohio', 'south_georgia_shag', 'red-tailed_hawk', 'bird', 'thayers_gull', 'silver_pheasant', 'meat-type_turkey', 'american_kestrel', 'great_horned_owl', 'pelecanus_occidentalis', 'red-breasted_merganser', 'northern_gannet', 'cat', 'lesser_snow_goose', 'barrow_s_goldeneye', 'colorado', 'american_black_duck', 'gray_gull', 'barn_owl', 'rt-hawk', 'dolphin', 'western_gull', 'backyard_chicken', 'pelecanus_thagus', 'eagle', 'falcon', 'domestic_turkey', 'chukar', 'chiloe_wigeon', 'california_gull', 'wildbird-fregata-magnificens', 'pigeon', 'royal_tern', 'california', "franklin's_gull", 'macaque', 'broiler_breeders_chicken', 'snowy_pl

In [12]:
# Ensure that user checks if there are any new animals

input("Check animals output. Afterwards, press ESCAPE to continue.")

''

## Merge all fastas into different files -- 8 segments * X genotypes ##

In [198]:
# 16 files needed
# huge_fasta = pd.DataFrame()

# for fastas in gisaid_dict.values():
#     # print(fastas.columns)
#     # print(len(fastas))
#     # break
#     # for f in fastas: # 16 files per batch 
#         # print(f)
#         # break 
#     print(fastas["Genotype"])
#     # Find Clade
#     metadata_fasta = pd.DataFrame()
#     for metadata in all_metadata_files:
#         metadata_fasta = pd.concat([metadata_fasta, metadata])
    
#     metadata_fasta["Identifier"] = metadata_fasta["Isolate_Id"]
#     metadata_fasta_concat = fastas.merge(metadata_fasta, on="Identifier", how="left")

#     huge_fasta = pd.concat([huge_fasta, metadata_fasta_concat])

# print(huge_fasta.columns)



# Now separate huge_fasta into genotypes * segments fastas
big_fastas = []

# print(huge_fasta)

# print(gisaid_dict)

# genotypes.append("Unassigned")
for gen in genotypes:
    print(gen)
    # for gisaid_fasta in gisaid_dict[gen]:
    
    # print(gen)
    for seg in unique_segments:
        print(seg)
        gisaid_fasta = gisaid_dict[gen + "_" + seg]
        # seg_specific_fasta = gisaid_fasta[gisaid_fasta["Segment"] == seg]
        print(gisaid_fasta)
        big_fastas.append(gisaid_fasta)

    print(gisaid_fasta[["Genotype"]])



B3.13
PB2
                                                  Header  Isolate_Id  \
2380   EPI_ISL_19792258|A/turkey/USA/007640-001/2025|...  007640-001   
2388   EPI_ISL_19743023|A/turkey/California/003574-00...  003574-001   
2396   EPI_ISL_19743022|A/turkey/California/003574-00...  003574-002   
2404   EPI_ISL_19743021|A/turkey/California/003574-00...  003574-003   
2412   EPI_ISL_19743020|A/turkey/California/003903-00...  003903-001   
...                                                  ...         ...   
18876  EPI_ISL_20266001|A/dairy_cow/California/026426...  026426-010   
18884  EPI_ISL_20265994|A/dairy_cow/California/02121-...   02121-002   
18892  EPI_ISL_20265992|A/dairy_cow/California/02121-...   02121-007   
18908  EPI_ISL_20265984|A/dairy_cow/California/02652-...   02652-001   
18916  EPI_ISL_20265578|A/dairy_cow/USA/024867-001/20...  024867-001   

                               Isolate_Name_x Subtype_x Segment  \
2380             A/turkey/USA/007640-001/2025      H5N1   

In [ ]:
# print(huge_fasta[huge_fasta["Genotype_x"] == "Not assigned"][["Identifier", "Genotype_x", "Genotype_y","Subtype_x"]]) # "Clade",  "Pathogenicity"]])

# Now that we have x fastas, write the files
for fasta in big_fastas:

    # Create a dictionary to create a file
    # fasta_df = fasta[["New_Name", "Sequence", "Clade"]]
    # print(fasta_df[fasta_df["Clade"].isna() == False])
    # break
    # fasta["New_Name_Clade_Path"] = fasta["New_Name"].apply(lambda x: x.split("\n")[0]) # + fasta["Clade"].apply(lambda x: "/" + x if x == x else "") + fasta["Pathogenicity"].apply(lambda x: "/" + x if x == x else "")
    # print(fasta_df["New_Name_Clade"])
    print("Original length:", len(fasta))
    fasta_dedup_ids = fasta.drop_duplicates(subset="Identifier", keep="last")
    # fasta = fasta_dedup_ids.drop_duplicates(subset="Partials", keep="last")
    print("New length:", len(fasta))
    # break 
    # fasta_df = fasta[["New_Name_Clade_Path", "Sequence"]]
    fasta_dict = pd.Series(fasta["Sequence"].values,index=fasta.full_header).to_dict()
    # print(fasta["Genotype"])
    # Create fasta file 
    try: 
        output_path = gisaid_files + fasta["Genotype"].values[0] + "_" + fasta["Segment"].values[0] + "_" + start_date + "--" + end_date + ".fasta" # Genotype and Segment should all be the same
        output_file = open(output_path, "w")
        for item in fasta_dict.keys():
            # print(item)
            value = fasta_dict[item] + "\n"
            # print(value)
            item = item.replace(" ", "_")
            
            # elif "EPI_ISL_20151596" in item:
            #     item = "EPI_ISL_20151596|A/Brown_Skua/Gough_Island/047354/2024|H5N1|Antarctica|2024-09-20|wild_avian|B3.2"
            output_file.write(item + "\n")
            output_file.write(value)
        print("Succeeded in finding results for genotype: ", fasta["Genotype"].values[0])
        output_file.close()
    except:
        # print(fasta["Genotype"])
        # print(fasta)
        print("Could not find any results for genotype.")
        # continue

# print(len(huge_fasta))
print(len(big_fastas[0]))

Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 1767
New length: 1767
Succeeded in finding results for genotype:  B3.13
Original length: 4256
New length: 4256
Succeeded in finding results for genotype:  D1.1
Original length: 4256
New length: 4256
Succeeded in finding results for genotype:  D1.1
Original length: 4256
New length: 4256
Succeeded in finding results for genotype:  D1.1
Original length: 4256
Ne

## Concatenate to Andersen_NCBI files and save

In [ ]:
# # Save metadata
# os.chdir(andersen_ncbi_virus)
# andersen_ncbi_virus_metadata = pd.read_csv("NCBI_Virus_Andersen_" + date_range + "_metadata.csv")
# andersen_ncbi_virus_metadata = andersen_ncbi_virus_metadata[["Identifier", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]
# andersen_ncbi_virus_metadata = andersen_ncbi_virus_metadata.reset_index(drop=True)



# # Rename columns
# metadata_fasta_csv = metadata_fasta_concat.rename(columns={"Isolate_Name_x":"Isolate", "Subtype_x":"Serotype", "Genotype_x":"Genotype", "Year":"Years", "Geo_Location":"Geo_Location_Abrv", "Date Collected":"Collection_Date", "Host_x":"Host", "Header":"Name"})
# metadata_fasta_csv = metadata_fasta_csv[["Identifier", "Host", "Collection_Date", "Isolate", "Serotype", "Segment", "Genotype", "Host_Type", "Years", "Geo_Location_Abrv", "Name"]]
# print(metadata_fasta_csv)
# metadata_fasta_csv["Isolate"] = metadata_fasta_csv["Isolate"].apply(lambda x: x.split("/")[-2] if len(x.split("/")) > 1 else x)
# segments_map = {"PB2":1, "PB1":2, "PA":3, "HA":4, "NP":5, "NA":6, "MP":7, "NS":8} # Name segments 
# metadata_fasta_csv["Segment"] = metadata_fasta_csv["Segment"].map(segments_map)
# metadata_fasta_csv = metadata_fasta_csv.reset_index(drop=True)

# print(andersen_ncbi_virus_metadata.columns)
# print(metadata_fasta_csv.columns)

# os.chdir(complete_files)
# metadata_all = pd.concat([andersen_ncbi_virus_metadata, metadata_fasta_csv], ignore_index=True)
# metadata_all.to_csv("combined_metadata.csv")

In [225]:
# Concat
os.chdir(complete_files)
segments = ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]

for gisaid_fasta_key in gisaid_dict:
    for andersen_ncbi_fasta_key in andersen_ncbi_genotypes:
        if gisaid_fasta_key == andersen_ncbi_fasta_key: # If the genotypes/segments are the same
            gisaid_fasta = gisaid_dict[gisaid_fasta_key]
            andersen_ncbi_fasta = andersen_ncbi_genotypes[andersen_ncbi_fasta_key]
            # print(gisaid_fasta)
            # print(andersen_ncbi_fasta)
            combined_fasta = pd.concat([gisaid_fasta, andersen_ncbi_fasta], ignore_index=True).drop_duplicates(subset="Partials", keep="last")
            combined_fasta = combined_fasta.rename(columns={"Sequence":"sequence"})
            combined_fasta["full_header"] = combined_fasta["full_header"].fillna(combined_fasta["Header"]).apply(lambda x: ">" + x if ">" not in x else x)
            print(combined_fasta)
            print(combined_fasta.columns)
            df_to_fasta(combined_fasta, gisaid_fasta_key + "_" + date_range + ".fasta", complete_files)

# common_genotypes = set()

# filenames_ncbi_andersen = []
# # File names NCBI_Virus_Andersen
# for dirpath, dirs, files in os.walk(andersen_ncbi_virus): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file) # Get file name
#         filenames_ncbi_andersen.append(file_name)
#         if ".csv" in file_name: # If metadata
#             metadata_ncbi_virus_andersen = pd.read_csv(file_name)
#             shutil.copy(file_name, complete_files)
#             # metadata_fasta_concat.to_csv("GISAID_metadata.csv")
#     break 

# filenames_gisaid = []
# # File names GISAID
# for dirpath, dirs, files in os.walk(gisaid_files): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file) # Get file name
#         filenames_gisaid.append(file_name)
#     break 

# for a_file in filenames_gisaid:
#     a_file_name = "_".join(a_file.split("/")[-1].split("_")[:2])
#     # print(a_file_name)
#     for nv_file in filenames_ncbi_andersen:
#         nv_file_name = "_".join(nv_file.split("/")[-1].split("_")[:2])
#         # print(nv_file_name)
#         if a_file_name == nv_file_name and ".fasta" in a_file:
#             common_genotypes.add(a_file_name)
#             filenames = [a_file, nv_file]
#             with open(complete_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile:
#                 for fname in filenames:
#                     with open(fname) as infile:
#                         for line in infile:
#                             outfile.write(line)
#                         infile.close()
#                 outfile.close()

# print(common_genotypes)

# for a_file in filenames_ncbi_andersen:
#     a_file_name = "_".join(a_file.split("/")[-1].split("_")[:2])
#     # If genotype not found in one of the datasets, include it as well
#     if a_file_name not in common_genotypes and ".fasta" in a_file:
#         print(a_file_name)
#         for segment in segments:
#             with open(complete_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile2:
#                 with open(a_file) as infile2:
#                     for line in infile2:
#                         outfile2.write(line)
#                     infile2.close()
#                 outfile2.close()

# for a_file in filenames_gisaid:
#     a_file_name = "_".join(a_file.split("/")[-1].split("_")[:2])
#     # If genotype not found in one of the datasets, include it as well
#     if a_file_name not in common_genotypes and ".fasta" in a_file:
#         print(a_file_name)
#         for segment in segments:
#             with open(complete_files + a_file_name + "_" + date_range + ".fasta", 'w') as outfile3:
#                 with open(a_file) as infile3:
#                     for line in infile3:
#                         outfile3.write(line)
#                     infile3.close()
#                 outfile3.close()

                                                 Header      Isolate_Id  \
132   EPI_ISL_19873945|A/swine/Kansas/ExpPig61AL5DPI...  ExpPig61AL5DPI   
133   EPI_ISL_19873944|A/swine/Kansas/ExpPig502DPI/2...    ExpPig502DPI   
136   EPI_ISL_19873941|A/swine/Kansas/ExpPig52AL3DPI...  ExpPig52AL3DPI   
137   EPI_ISL_19873940|A/swine/Kansas/ExpPig51AL5DPI...  ExpPig51AL5DPI   
138   EPI_ISL_19873939|A/swine/Kansas/ExpPig50AL3DPI...  ExpPig50AL3DPI   
...                                                 ...             ...   
6897  GCA_054872375|A/Bovine/California/B2400548/202...        B2400548   
6898  GCA_054872415|A/Bovine/California/B2401408/202...        B2401408   
6899  GCA_054872425|A/Bovine/California/B2401620/202...        B2401620   
6900  GCA_054872495|A/Bovine/California/B2401751/202...        B2401751   
6901  GCA_054872785|A/Turkey/California/B2402661/202...        B2402661   

                          Isolate_Name_x Subtype_x Segment Location_Header  \
132   A/swine/Kansas/

In [226]:
# Create new labels FOR D1.3 ONLY

# Create new labels
os.chdir(home)
new_county_info = pd.read_excel("Positive Case Info  Summary (version 1).xlsx")
og_county_info = pd.read_csv("Positive Case Info = Summary Counties.csv")
sra_info = pd.read_excel("OH-IN_poultry_clade_matched_seq_metadata (version 1).xlsx")

sra_info["Event ID"] = sra_info["SRA Event ID"]

# print(sra_info)
# print(og_county_info["County"])
# print(county_info["HPAI Case Description"])
# print(county_info["Event ID"])

med_county_info = new_county_info.merge(sra_info, on=["Event ID"])

county_info = med_county_info.merge(og_county_info, on=["Event ID", "Site Owner", "Collection Date"], how="left")
# print(county_info)
# print(county_info["County"])
# new_county_info["County"] = np.where(new_county_info["County"] == "",  new_county_info["County"])
county_info["County"] = county_info["HPAI Case Description"].apply(lambda x: # og_county_info.loc[county_info["HPAI Case Description"].str.contains('_'.join(x.split(' ')[:-1])), 'County'] if len(og_county_info.loc[county_info["HPAI Case Description"].str.contains('_'.join(x.split(' ')[:-1])), 'County']) > 0 else 
                                                                   '_'.join(x.split(' ')[:-1]) + "_County_" + og_county_info[og_county_info["County"].str.contains("_".join(x.split(' ')[:-1]))]["County"].values[0].split("_")[-1] if x == x else x)
# print(county_info["Collection Date"])

# print(county_info[county_info["County"].notna()]["County"])

label_mapping = {}
for id in county_info["SRA Accession"].values:
    print(id)
    if id == id and county_info[county_info["SRA Accession"] == id]["County"].values[0] == county_info[county_info["SRA Accession"] == id]["County"].values[0]: # If not nan
        label_mapping[id] = county_info[county_info["SRA Accession"] == id]["County"].iloc[0] + "|" + county_info[county_info["SRA Accession"] == id]["Site Owner"].iloc[0].replace(" ", "_")

print(len(label_mapping))
print(label_mapping)

for dirpath, dirs, files in os.walk(complete_files):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if "D1.3" in file_name:
            output_file = open(file_name, "r+")
            with open(file_name) as f:
                lines = f.readlines()
                for line in lines:
                    for key in label_mapping:
                        # break 
                        if key in line and len(line.split("|")) < 8: 
                            print(line.split("|")[-3])
                            if dateutil.parser.parse(line.split("|")[-3], default=dateutil.parser.parse("2000-01-01")).month == dateutil.parser.parse("2000-01-01").month and dateutil.parser.parse(line.split("|")[-3], default=dateutil.parser.parse("2000-01-01")).day == dateutil.parser.parse("2000-01-01").day: # If no collection date in line
                                date = county_info[county_info["SRA Accession"] == key]["Collection Date"].apply(lambda x: dateutil.parser.parse(str(x))).values[0]
                                # print(date)
                                split_line = line.split("|")
                                split_line[-3] = dateutil.parser.parse(str(date)).strftime("%Y-%m-%d")
                                line = ("|").join(split_line)
                                print(line)
                                print("date changed")
                            if line.split("|")[3] == "USA":
                                split_line = line.split("|")
                                split_line[3] = "USA-" + county_info[county_info["SRA Accession"] == key]["County"].apply(lambda x: x.split("_")[-1]).values[0]
                                line = ("|").join(split_line)
                            line = line.replace("\n", "") + "|" + label_mapping[key] + "\n"
                            print(line)
                            print("changed")
                    output_file.write(line)
                f.close()
            output_file.close()
    break 

SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32654191
SRR32654192
SRR32654193
SRR32654195
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR32415174
SRR32415176
SRR32415177
SRR32415178
SRR31959950
SRR31959951
SRR31959950
SRR31959951
SRR32125543
SRR32125544
SRR32125543
SRR32125544
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226901
SRR32226902
SRR32226904
SRR32226915
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226934
SRR32226935
SRR32226936
SRR32226938
SRR32226929
SRR32226930
SRR32226929
SRR32226930
SRR32226927
SRR32226928
SRR32226927
SRR32226928
SRR32254627
SRR32254628
SRR32254627
SRR3